# W5C1: From a crime scene to a search engine

Run every cell from the top. **Everything already works.**

**A body in Lauriston Gardens, and six things noted at the scene.**

Baker Street keeps a casebook: a hundred prior cases, each one a list of what was observed at it. Somewhere in it is the case that most resembles tonight's, and Holmes wants it before the fog lifts.

The scene is six ordinary words. That is the whole difficulty.

Today you will:

1. Watch Ctrl+F fail at a question you actually have.
2. Build **tf**, then **idf**, then **cosine similarity**, by hand.
3. Point the result at a hundred case files and name the case.

Nothing to submit. Answers are in the last cell.

## Part 0. Why not just search for it?

You have six observations and a hundred case files. Try the obvious thing first.

In [ ]:
# Setup. Run this cell first.
import numpy as np
import pandas as pd

casebook = pd.read_csv("data/casebook.csv")

print(casebook.shape[0], "case files")
print()
print(casebook.head(4).to_string(index=False))

In [ ]:
SCENE = "mud rain tobacco cab boots bruise"

print("Ctrl+F for the whole scene:",
      int(casebook["observations"].str.contains(SCENE).sum()), "cases")
print()
print("Ctrl+F for one observation at a time:")
for clue in SCENE.split():
    hits = int(casebook["observations"].str.contains(clue).sum())
    print(f"   {clue:9s} {hits:3d} cases")

has_everything = casebook["observations"].apply(
    lambda text: all(word in text for word in SCENE.split()))
print()
print("cases noting all six:", int(has_everything.sum()))
print()
print("No case has all six, and each one alone returns a pile. There is no")
print("string to find. The question is which case is MOST like the scene,")
print("counting a rare observation for more than one that is everywhere.")

## Part 1. tf: how much of this case file is that observation?

Four case files, small enough that every number fits on the screen.

In [ ]:
# Four tiny case files, so every number fits on the screen.
DOCS = [
    "creosote boots mud rope london tar",
    "ledger boots tobacco london gaslight",
    "boots bruise rain london mud",
    "cab boots mud london rain bruise",
]
TITLES = ["the riverside warehouse", "the pawnbroker's back room",
          "the area railings", "the cabman's testimony"]

for number, (title, document) in enumerate(zip(TITLES, DOCS), 1):
    print(f"case {number}  {title:28s} {document}")

In [ ]:
# The vocabulary: every distinct observation, in a fixed order. Slot i is the
# same observation in every row, which is the only reason two rows compare.
VOCAB = []
for document in DOCS:
    for word in document.split():
        if word not in VOCAB:
            VOCAB.append(word)
VOCAB = sorted(VOCAB)

print(len(VOCAB), "distinct observations")
print(VOCAB)

In [ ]:
# Count the words of one case file into an array, then divide by its length.
def term_frequencies(document):
    """One row of tf values, one slot per word in VOCAB."""
    words = document.split()
    counts = np.zeros(len(VOCAB))
    for word in words:
        if word in VOCAB:          # a scene may hold things never seen before
            counts[VOCAB.index(word)] = counts[VOCAB.index(word)] + 1
    return counts / len(words)


print("case 1:", DOCS[0])
print()
print(pd.Series(term_frequencies(DOCS[0]), index=VOCAB).round(3).to_string())
print()
print("Every observation scores 0.167. tf rates london exactly as highly as")
print("creosote, which is the problem Part 2 fixes.")

In [ ]:
# ================== TRY IT 1 ==================
# Is `boots` worth the same tf in all four case files? Why?
# ==============================================


## Part 2. idf: how much does this observation narrow it down?

An observation that turns up at every crime tells you nothing about which one
you are looking at.

In [ ]:
# One count per vocabulary word: how many case files mention it at all.
def document_frequency(word):
    """In how many of the four case files does this observation appear?"""
    return sum(1 for document in DOCS if word in document.split())


def inverse_document_frequency(word):
    """log(total cases / cases mentioning it). In all of them -> exactly 0."""
    return np.log(len(DOCS) / document_frequency(word))


print(f"{'observation':12s} {'in':>3}  {'idf':>6}")
for word in VOCAB:
    print(f"{word:12s} {document_frequency(word):>3}  "
          f"{inverse_document_frequency(word):>6.3f}")
print()
print("boots and london are in all four, so each divides by itself, the log is")
print("zero, and both are switched OFF. Not filtered by a list of boring words.")
print("Switched off by the arithmetic.")

In [ ]:
# tf-idf is the two arrays multiplied slot by slot: (12,) * (12,) -> (12,).
def tf_idf(document):
    """One row of tf-idf weights for a case file, or for a scene."""
    weights = np.array([inverse_document_frequency(word) for word in VOCAB])
    return term_frequencies(document) * weights


print("case 1:", DOCS[0])
print()
print(pd.Series(tf_idf(DOCS[0]), index=VOCAB).round(4).to_string())
print()
print("creosote 0.2310 at the top; boots and london exactly 0.0000. The only")
print("observation here that tells you WHICH case this is, is the rare one.")

In [ ]:
# ================== TRY IT 2 ==================
# Which observation is heaviest in case 2, and what is `london` worth there?
# ==============================================


## Part 3. Cosine similarity: compare directions, not lengths

A nine-observation case and a three-observation case about the same crime sit
far apart by straight-line distance, purely because one has more numbers in it.

In [ ]:
# Cosine similarity: a dot product over two lengths. NumPy has all three.
def cosine(first, second):
    """The cosine of the angle between two tf-idf rows."""
    lengths = np.linalg.norm(first) * np.linalg.norm(second)
    if lengths == 0:
        return 0.0
    return np.dot(first, second) / lengths


# Stack the four rows into one array: 4 cases x 12 observations.
all_weights = np.array([tf_idf(document) for document in DOCS])
print("all_weights.shape:", all_weights.shape)
print()

scores = np.zeros((4, 4))
for i in range(4):
    for j in range(4):
        scores[i][j] = cosine(all_weights[i], all_weights[j])

labels = [f"case {n}" for n in range(1, 5)]
print(pd.DataFrame(scores.round(3), index=labels, columns=labels).to_string())

In [ ]:
# A SCENE is just another short case file. Weight it the same way and compare.
scene = "mud rain bruise"

scene_weights = tf_idf(scene)

print("scene:", scene)
print()
for number in range(4):
    print(f"  case {number + 1}  {cosine(scene_weights, all_weights[number]):.3f}"
          f"   {TITLES[number]}")
print()
print("That is a search engine. Everything after this is the same idea, faster.")

In [ ]:
# ================== TRY IT 3 ==================
# Search the four cases for `boots london`. Would you trust the ranking?
# ==============================================


## Part 4. The whole casebook

Same arithmetic, a hundred case files, and a library that does it in one line.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Only the OBSERVATIONS are indexed, never the title. You search observations
# and get back a case name, so no answer is something you could have searched.
vectorizer = TfidfVectorizer()
case_vectors = vectorizer.fit_transform(casebook["observations"])

print("case_vectors.shape:", case_vectors.shape)
print("   ", case_vectors.shape[0], "cases x",
      case_vectors.shape[1], "distinct observations")
print()

# A scene becomes a vector with the SAME vocabulary. transform, not fit.
scene_vector = vectorizer.transform([SCENE])
print("the scene becomes", scene_vector.shape)

In [ ]:
# What every observation is worth, measured over all 100 case files.
weights = pd.Series(vectorizer.idf_, index=vectorizer.get_feature_names_out())

print("noted at almost every crime, so worth almost nothing:")
print(weights.sort_values().head(5).round(3).to_string())
print()
print("noted at two or three, so decisive:")
for clue in ["creosote", "blowpipe", "cipher", "laudanum", "gaslight"]:
    print(f"{clue:12s} {weights[clue]:.3f}")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# One call scores the scene against all 100 case files.
scene_scores = cosine_similarity(scene_vector, case_vectors)[0]

ranked = pd.DataFrame({"title": casebook["title"], "score": scene_scores})
ranked = ranked.sort_values("score", ascending=False)

print("SCENE:", SCENE)
print()
print(ranked.head(5).to_string(index=False, float_format=lambda v: f"{v:.3f}"))

---

## After the break: the casebook race

Three questions, one shared answer per team. The first team with **a case, an
observation, and a reason** wins; the reason is the part that counts.

In [ ]:
# GIVEN. `search` scores exactly ONE case file, so it always names the same one.
# Fixing that is YOUR TURN 1.
def search(query, top=5):
    """Rank the casebook against a query. Returns the best `top` cases."""
    query_vector = vectorizer.transform([query])
    scores = np.zeros(len(casebook))
    for i in range(1):                      # <-- YOUR TURN 1 is here
        scores[i] = cosine_similarity(query_vector, case_vectors[i])[0][0]
    out = pd.DataFrame({"title": casebook["title"], "score": scores})
    return out.sort_values("score", ascending=False).head(top)


print(search(SCENE).to_string(index=False, float_format=lambda v: f"{v:.3f}"))

In [ ]:
# ================== YOUR TURN 1 ==================
# `search` only ever scores the first case file. Make it score all of them.
#
# Expected: The Brixton Road affair 0.687, then The stolen greatcoat 0.626 and
#           The cabman's testimony 0.619. Both a Python loop and the one-shot
#           cosine_similarity(query_vector, case_vectors) are accepted; the
#           one-shot is about a hundred times faster.
# =================================================
# Edit the loop in the GIVEN cell above, then run this.
print(search(SCENE).to_string(index=False, float_format=lambda v: f"{v:.3f}"))

In [ ]:
# ================== YOUR TURN 2 ==================
# Holmes can send Wiggins back to Lauriston Gardens for ONE more
# observation. Two different answers are worth having:
#   a. which single observation raises your confidence the most?
#   b. which single observation CHANGES which case you chase?
# Then: which of the six you already have is doing the least work?
#
# Expected: a. gaslight -> The Brixton Road affair 0.938, the same case, harder.
#           b. creosote -> The riverside warehouse takes the lead at 0.580,
#              a different case entirely, and the lower score is not a worse
#              answer: a longer query is divided by a longer length.
#           c. boots, idf 1.744, noted at 47 of the 100 crimes. cab is rarest
#              at 2.480, and it is doing the most.
# =================================================
EXTRA = "gaslight"        # <-- try others

print(search(SCENE + " " + EXTRA).to_string(index=False,
      float_format=lambda v: f"{v:.3f}"))
print()
for clue in SCENE.split():
    print(f"  {clue:9s} idf {weights[clue]:.3f}")

## Answers

Try each task before reading.

In [ ]:
# TRY IT 1
#   No. boots is in all four, but tf divides by the case file's LENGTH:
#   0.167, 0.200, 0.200, 0.167, because the files are 6, 5, 5 and 6 long.
#   tf is a share of the file, not a count.

# TRY IT 2
#   ledger and tobacco and gaslight all tie at 0.2773, each noted at one case
#   only. london is 0.0000, and so is boots. The tie is the honest answer: with
#   four documents there is not much to separate three unique observations.

# TRY IT 3
#   0.000 for every case. Both words are in all four files, so both have idf 0,
#   the query vector is all zeros, and cosine returns 0.0 by the guard clause.
#   A query made only of ubiquitous words carries no information at all. On the
#   full casebook the same query returns a confident-looking order built on
#   almost nothing, which is worse, because it looks like an answer.

# YOUR TURN 1
#   for i in range(len(casebook)):
#       scores[i] = cosine_similarity(query_vector, case_vectors[i])[0][0]
#
#   or, once, without the loop:
#       scores = cosine_similarity(query_vector, case_vectors)[0]
#
#   The Brixton Road affair  0.687
#   The stolen greatcoat     0.626
#   The cabman's testimony   0.619

# YOUR TURN 2
#   a. gaslight. 0.938, and The Brixton Road affair stays on top. Checked over
#      the WHOLE vocabulary, not just a few guesses, so brute force finds it too.
#   b. creosote. The riverside warehouse takes the lead at 0.580. A different
#      case, a different theory of the crime, and the SCORE WENT DOWN, because
#      a seventh word lengthens the query vector and cosine divides by length.
#      A longer query is not a better query.
#   c. boots, idf 1.744, noted at 47 of 100 crimes. Then bruise 1.786 and
#      mud 2.060. cab is the rarest of the six at 2.480.
#
#   The two answers are different kinds of answer. (a) makes you more sure of
#   what you already thought. (b) changes your mind. Holmes wants (b).

# The three things worth carrying out of today:
#   1. Exact matching answers "where is this string". Ranking answers "which of
#      these is most like what I have", and only the second is usually the
#      question.
#   2. idf is a weighting, not a stop-word list. An observation at every crime
#      gets zero by arithmetic.
#   3. A search engine always returns an order. Whether the order means anything
#      depends on whether your words were rare.